# Withdrawal risk at an actionable landmark

At the start of a decision day, rank known registrations with no recorded earlier unregistration. The target is the final official Withdrawn outcome. A withdrawal on the decision day is still in the risk set. Missing registration dates are excluded because membership cannot be established. A missing unregistration date means no observed exit, irrespective of the eventual label.

The protocol compares days 14, 28 and 42 and fixes the rule, estimators, feature sets and selection criterion before scoring 2014J. Early clicks and completion for all entrants answer a different question. Below we measure both the endpoint change and how removing known exits affects separation on development data.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import calendar
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score

SEED = 42
RAW = Path("../data/raw")
KEY = ["code_module", "code_presentation", "id_student"]
TERM = "code_presentation"
MODULE = KEY[:2]
DEV = ["2013B", "2013J", "2014B"]
HOLDOUT = "2014J"
DAYS = [14, 28, 42]
CATS = ["code_module", "season", "age_band", "highest_education", "imd_band",
        "gender", "disability", "region"]
STATIC = ["num_of_prev_attempts", "studied_credits", "date_registration", "prior_records"]

def read_terms(name, terms):
    table = pd.read_csv(RAW / (name + ".csv"))
    return table.loc[table[TERM].isin(terms)].copy()

def inputs(terms):
    info = read_terms("studentInfo", terms)
    reg = read_terms("studentRegistration", terms)
    students = info.merge(reg, on=KEY, validate="one_to_one")
    assert len(students) == len(info) == len(reg)
    students["withdrawn"] = students.final_result.eq("Withdrawn").astype(int)
    chunks = []
    for events in pd.read_csv(RAW / "studentVle.csv", chunksize=400_000):
        keep = events[TERM].isin(terms) & events.date.between(0, max(DAYS)-1) & events.sum_click.gt(0)
        chunks.append(events.loc[keep])
    activity = pd.concat(chunks, ignore_index=True).merge(
        read_terms("vle", terms)[MODULE + ["id_site", "activity_type"]],
        on=MODULE + ["id_site"], validate="many_to_one", how="left")
    assert activity.activity_type.notna().all()
    schedule = read_terms("assessments", terms).dropna(subset=["date"])
    schedule = schedule.loc[schedule.assessment_type.ne("Exam")].sort_values(["date", "id_assessment"])
    first = schedule.drop_duplicates(MODULE)[MODULE + ["id_assessment", "date"]]
    first = first.rename(columns={"date": "due"})
    submissions = pd.read_csv(RAW / "studentAssessment.csv",
                              usecols=["id_student", "id_assessment", "date_submitted", "is_banked"])
    first_sub = submissions.loc[submissions.is_banked.eq(0)].merge(first, on="id_assessment")
    return students, activity, first, first_sub

def start_bound(term, latest=False):
    year = int(term[:4])
    month = {"B": 2, "J": 10}[term[-1]]
    day = calendar.monthrange(year, month)[1] if latest else 1
    return pd.Timestamp(year, month, day)

dev, activity, first, first_sub = inputs(DEV)
# A count of earlier registrations only. Even their latest possible calendar date
# must precede the current term's earliest start. Include overlapping courses, never outcomes.
history = dev[KEY + ["date_registration"]].dropna().copy()
history["registered_by"] = (history[TERM].map(lambda t: start_bound(t, True))
                            + pd.to_timedelta(history.date_registration, unit="D"))

## Feature and outcome availability

| Input | Snapshot at the start of day d |
| --- | --- |
| Module, intake season, demographics, credits, previous attempts | Enrollment attributes as released. We assume these fields were recorded at entry. Missing categories are explicit. IDs only group records. |
| Registration day and prior record count | Require registration strictly before d. Count dated records in earlier presentations known by the current start. Never use their final results. |
| Weekly click sums, last-week change, active days, unique sites, distinct activity types, time since last click | Positive VLE activity on days 0 through d−1 inclusive. Use d for recency if never active. Log-transform count features. |
| First assignment | First scheduled non-exam with a known due day, breaking due-date ties by assessment ID. Use its published deadline and non-banked submissions dated before d, including early submissions. Distinguish submitted, submitted on time and overdue with no submission. Marks and returned grades have no role. |
| Rule | No positive clicks on days 0–13. Estimate its two probabilities on the training fold. Equal-score reviews use a seeded, outcome-independent order. |
| Eligibility and target | Unregistration is used solely to remove exits already observed at d. Final status supplies the label. All transforms with learned parameters are fit on training rows. |

The source identifies the start month rather than a calendar date. A training module must have ended before the earliest possible scoring date. Use the last possible start date plus the advertised duration as an embargo. For mature training and validation presentations, reconstruct the administrative withdrawal label from exit dates strictly before the evaluation clock. This prevents even late administrative exits from becoming early labels. The official final outcome is reserved for the terminal evaluation; development concordance is reported below. Publication times for final labels and edits to entry attributes are unavailable.

In [ ]:
def snapshot(students, events, first, submissions, day):
    known = students.date_registration.lt(day)
    active = students.date_unregistration.isna() | students.date_unregistration.ge(day)
    base = students.loc[known & active].copy()
    base["season"] = base[TERM].str[-1]
    for term in base[TERM].unique():
        past = history.loc[(history[TERM] < term) & (history.registered_by < start_bound(term))]
        count = past.groupby("id_student").size()
        current = base[TERM].eq(term)
        base.loc[current, "prior_records"] = base.loc[current, "id_student"].map(count).fillna(0)
    seen = events.loc[events.date.lt(day)].copy()
    assert seen.date.between(0, day-1).all()
    seen["week"] = seen.date // 7
    grouped = seen.groupby(KEY)
    counts = grouped.agg(active_days=("date", "nunique"), sites=("id_site", "nunique"),
                         types=("activity_type", "nunique"), last=("date", "max"))
    weekly = seen.groupby(KEY + ["week"]).sum_click.sum().unstack(fill_value=0)
    weeks = [f"week_{n}" for n in range(day//7)]
    weekly = weekly.reindex(columns=range(len(weeks)), fill_value=0)
    weekly.columns = weeks
    base = base.join(counts, on=KEY).join(weekly, on=KEY)
    volumes = weeks + ["active_days", "sites", "types"]
    base[volumes] = base[volumes].fillna(0)
    base["no_click14"] = base[weeks[:2]].sum(axis=1).eq(0).astype(int)
    base["recency"] = day - 1 - base["last"].fillna(-1)
    base[volumes] = np.log1p(base[volumes])
    base["week_change"] = base[weeks[-1]] - base[weeks[-2]]
    observed = submissions.loc[submissions.date_submitted.lt(day), KEY + ["date_submitted"]]
    base = base.merge(first[MODULE + ["due"]], on=MODULE, how="left", validate="many_to_one")
    base = base.merge(observed, on=KEY, how="left", validate="one_to_one")
    base["submitted"] = base.date_submitted.notna().astype(int)
    base["on_time"] = base.date_submitted.le(base.due).astype(int)
    base["overdue"] = (base.due.lt(day) & base.submitted.eq(0)).astype(int)
    return base

frames = {d: snapshot(dev, activity, first, first_sub, d) for d in DAYS}
ever = activity.loc[activity.date.lt(14), KEY].drop_duplicates()
diagnostic = dev.merge(ever.assign(clicked=True), on=KEY, how="left", validate="one_to_one")
diagnostic["no_click14"] = diagnostic.clicked.isna().astype(int)
def describe(frame, group):
    return (frame.assign(completed=frame.final_result.isin(["Pass", "Distinction"]))
            .groupby("no_click14").agg(n=("withdrawn", "size"), withdrawal=("withdrawn", "mean"),
                                       completion=("completed", "mean")).assign(population=group))
print(pd.concat([describe(diagnostic, "all development enrollments")] +
                [describe(frames[d], f"eligible day {d}") for d in DAYS]).round(3).to_string())
del activity

In [ ]:
courses = read_terms("courses", DEV).set_index(MODULE)
latest = pd.Series([start_bound(t, True) for _, t in courses.index], index=courses.index)
ready = latest + pd.to_timedelta(courses.module_presentation_length, unit="D")

def mature(frame, term, day):
    as_of = start_bound(term) + pd.Timedelta(days=day)
    mask = (frame[TERM] < term) & pd.MultiIndex.from_frame(frame[MODULE]).isin(ready[ready < as_of].index)
    closed = frame.loc[mask].copy()
    observed_days = (as_of - pd.to_datetime(closed[TERM].map(lambda t: start_bound(t, True)))).dt.days
    closed["withdrawn"] = closed.date_unregistration.lt(observed_days).astype(int)
    return closed

def numeric_features(day, spec):
    weeks = [f"week_{w}" for w in range(day//7)]
    if spec == "clicks":
        return weeks + ["active_days", "sites"]
    return weeks + ["active_days", "sites", "types", "recency", "week_change",
                    "due", "submitted", "on_time", "overdue"] + STATIC

def predict(train, valid, day, name, spec):
    if name == "Rule":
        rates = train.groupby("no_click14").withdrawn.mean()
        return valid.no_click14.map(rates).fillna(train.withdrawn.mean()).to_numpy()
    nums = numeric_features(day, spec)
    transformers = [("numeric", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), nums)]
    if spec == "full":
        categorical = make_pipeline(SimpleImputer(strategy="constant", fill_value="Missing"),
                                   OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        transformers.append(("category", categorical, CATS))
    estimator = (LogisticRegression(C=1, max_iter=500, random_state=SEED) if name == "Logistic"
                 else HistGradientBoostingClassifier(max_iter=120, max_leaf_nodes=15,
                                                     learning_rate=0.06, l2_regularization=1,
                                                     early_stopping=False, random_state=SEED))
    model = make_pipeline(ColumnTransformer(transformers), estimator)
    model.fit(train, train.withdrawn)
    return model.predict_proba(valid)[:, 1]

def metrics(y, prob):
    # Rows have a seeded tie order before evaluation; stable sorting preserves it.
    k = int(np.ceil(len(y)/10))
    top = np.argsort(-prob, kind="stable")[:k]
    precision = y[top].mean()
    return {"AUC": roc_auc_score(y, prob), "AP": average_precision_score(y, prob),
            "P@10%": precision, "lift": precision/y.mean(), "Brier": brier_score_loss(y, prob)}

SPECS = [("Rule", "binary")] + [(name, spec) for name in ["Logistic", "Boosting"]
                                for spec in ["clicks", "full"]]

## Rolling-origin selection

Evaluate each candidate on each later development intake with mature training data. Validation labels must also have matured before the candidate's prospective 2014J decision date. The decision requires two usable validation intakes. An unavailable fold is printed explicitly. Select highest equal-fold mean average precision, then earlier day, then the fixed candidate order. The fold tables include the risk-set size and prevalence because later decisions leave less intervention time and change who is eligible. Baseline probabilities, imputation, encoding and models are refit at each origin.

In [ ]:
rows = []
for day, frame in frames.items():
    available = mature(frame, HOLDOUT, day)
    for term in DEV[1:]:
        train = mature(frame, term, day)
        valid = available.loc[available[TERM].eq(term)].sample(frac=1, random_state=SEED)
        if train.empty or valid.empty:
            print(f"Day {day}, {term}: unavailable (train {len(train)}, mature validation {len(valid)})")
            continue
        for name, spec in SPECS:
            p = predict(train, valid, day, name, spec)
            rows.append(dict(day=day, valid=term, train_n=len(train), n=len(valid),
                             prevalence=valid.withdrawn.mean(), model=name, features=spec,
                             **metrics(valid.withdrawn.to_numpy(), p)))
cv = pd.DataFrame(rows)
print(cv.round(4).to_string(index=False))
summary = cv.groupby(["day", "model", "features"], sort=False).agg(
    folds=("valid", "nunique"), AP=("AP", "mean"), AUC=("AUC", "mean"), lift=("lift", "mean")).reset_index()
ranked = summary.loc[summary.folds.eq(2)].sort_values(["AP", "day"], ascending=[False, True], kind="stable")
winner = ranked.iloc[0]
day = int(winner.day)
print(summary.round(4).to_string(index=False))
print(f"LOCKED: day {day}, {winner.model}, {winner.features}, mean AP {winner.AP:.4f}")

In [ ]:
# A common development origin permits a like-for-like timing comparison including day 14.
# Read its outcomes at the later development freeze. This diagnostic does not enter selection.
rows = []
for d in DAYS:
    hist = mature(frames[d], "2014B", d)
    later = mature(frames[d], HOLDOUT, max(DAYS))
    val = later.loc[later[TERM].eq("2014B")].sample(frac=1, random_state=SEED)
    for name, spec in [("Rule", "binary"), ("Logistic", "full"), ("Boosting", "full")]:
        p = predict(hist, val, d, name, spec)
        rows.append(dict(day=d, model=name, n=len(val), prevalence=val.withdrawn.mean(),
                         **metrics(val.withdrawn.to_numpy(), p)))
print("2013B training, 2014B validation, common maturity clock")
print(pd.DataFrame(rows).round(4).to_string(index=False))
historical = mature(frames[day], HOLDOUT, day)
print("Training label audit (dated exit vs official withdrawal):")
print(pd.crosstab(historical.withdrawn, historical.final_result.eq("Withdrawn")).to_string())

## Locked evaluation

Open the holdout for this decision day once. At this same landmark report the trained rule and the validation-selected feature set for each model family. Resample student IDs with replacement, keeping all their enrollments together. Each replicate recomputes the review budget, ranking, prevalence and metrics. Intervals condition on the fitted models and this intake. The paired difference uses the same resamples as the rule. The reliability table uses quantile bins, merging ties.

In [ ]:
finalists = ranked.loc[ranked.day.eq(day)].drop_duplicates("model")
train = mature(frames[day], HOLDOUT, day)
test_inputs = inputs([HOLDOUT])
test = snapshot(*test_inputs, day).sample(frac=1, random_state=SEED)
del test_inputs
pred = {row.model: predict(train, test, day, row.model, row.features)
        for row in finalists.itertuples()}
y = test.withdrawn.to_numpy()
print(f"Day {day}; training {len(train):,}; holdout {len(test):,}; students {test.id_student.nunique():,}; "
      f"withdrawals {y.sum():,}; prevalence {y.mean():.4f}; review budget {int(np.ceil(len(y)/10)):,}")
point = pd.DataFrame({name: metrics(y, p) for name, p in pred.items()}).T
print(point.round(4).to_string())
k = int(np.ceil(len(y)/10))
for name, prob in pred.items():
    queue = np.argsort(-prob, kind="stable")[:k]
    print(f"{name}: {y[queue].sum()} confirmed withdrawals among {k} reviews; "
          f"mean probability {prob.mean():.4f}, observed {y.mean():.4f}")

In [ ]:
groups = list(test.reset_index(drop=True).groupby("id_student", sort=False).indices.values())
rng = np.random.default_rng(SEED)
draws = []
for rep in range(1000):
    idx = np.concatenate([groups[g] for g in rng.integers(len(groups), size=len(groups))])
    # Stable, fixed, outcome-independent tie order for each resampled queue.
    idx.sort()
    for name, p in pred.items():
        draws.append(dict(rep=rep, model=name, **metrics(y[idx], p[idx])))
boot = pd.DataFrame(draws)
bounds = boot.groupby("model")[list(point)].quantile([0.025, 0.975])
print("Student-cluster percentile intervals")
print(bounds.round(4).to_string())
paired = boot.pivot(index="rep", columns="model", values="AP")
print("Selected minus rule AP:", round(point.loc[winner.model,"AP"] - point.loc["Rule","AP"], 4),
      "95% CI", (paired[winner.model] - paired.Rule).quantile([.025, .975]).round(4).tolist())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), layout="constrained")
ax.plot([0, 1], [0, 1], "--", color="grey")
for name, p in pred.items():
    bins = pd.qcut(p, 5, duplicates="drop") if np.unique(p).size > 5 else p
    reliability = pd.DataFrame({"bin": bins, "p": p, "outcome": y}).groupby("bin", observed=True).agg(
        n=("p", "size"), predicted=("p", "mean"), observed=("outcome", "mean"))
    print(name, reliability.round(3).to_string(index=False), sep="\n")
    ax.plot(reliability.predicted, reliability.observed, "o-", label=name)
ax.set(xlabel="Mean predicted withdrawal probability", ylabel="Observed withdrawal fraction")
ax.legend()
fig.savefig("../assets/05_withdrawal_calibration.png", dpi=125)
plt.show()